<div align="center">
  <h3><b>ESCUELA POLITÉCNICA NACIONAL</b></h3>
  <h3><b>FACULTAD DE INGENIERÍA EN SISTEMAS</b></h3>
  <h3><b>INGENIERÍA EN CIENCIAS DE LA COMPUTACIÓN</b></h3>
  <h3><b>RECUPERACIÓN DE LA INFORMACIÓN</b></h3>
</div>

---
**Nombre**   Mark Hernández        
**Fecha**    04/05/26  
**Docente**  Iván Carrera

# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

Primero obtenemos el corpus del archivo .txt

In [6]:
import os
import re

path = "..\\03tfidf\\data"
file = os.listdir(path)

contenido = []

with open(os.path.join(path, file[0]),"r", encoding="utf-8") as f:
    contenido = f.read().lower()
    palabras = re.findall(r'\w+', contenido)
    vocabulario = set(palabras)

print(list(vocabulario)[:100])

print(len(vocabulario))


['avistamiento', 'laguna', 'sorprende', 'patrimonio', 'como', 'baños', 'disfrutan', 'invita', 'famosas', 'playas', 'volcán', 'a', 'fiestas', 'una', 'tiene', 'naturaleza', 'ruta', 'lugar', 'para', 'de', 'llena', 'famoso', 'interesados', 'color', 'colonial', 'ecuador', 'típica', 'entre', 'agua', 'santa', 'ofrece', 'histórico', 'del', 'turquesa', 'tranquilo', 'arquitectura', 'atrae', 'quito', 'islas', 'rafting', 'espectacular', 'atraen', 'se', 'ecuatoriana', 'caminatas', 'muchos', 'vilcabamba', 'megadiverso', 'aves', 'gastronomía', 'turismo', 'es', 'artesanía', 'cotopaxi', 'turistas', 'longevidad', 'galápagos', 'feriados', 'cajas', 'malecón', 'ideal', 'lagunas', 'centro', 'la', 'única', 'increíble', 'perfecto', 'inolvidable', 'mercado', 'surf', 'amazonía', 'deslumbra', 'humanidad', 'local', 'experiencia', 'canopy', 'sorprendente', 'deportes', 'país', 'su', 'visitar', 'y', 'biodiversidad', 'en', 'con', 'nacional', 'feriado', 'otavalo', 'spondylus', 'aventura', 'ecológico', 'un', 'conecta',

Luego procedemos a calcular los valores de frecuencia con que aparece cada término dentro del documento. Para ello hacemos uso del concepto de índice invertido.

In [7]:
import pandas as pd

indice_invertido = {}

for palabra in vocabulario:
    indice_invertido[palabra] = 0

for palabra in re.findall(r'\w+', contenido): 
    if palabra in indice_invertido.keys():
        indice_invertido[palabra] += 1

print(len(indice_invertido))
    
t1 = pd.DataFrame(list(indice_invertido.items()), columns=["palabra", "frecuencia"])
t1.head(20)

118


,palabra,frecuencia
0,avistamiento,44
1,laguna,32
2,sorprende,33
3,patrimonio,20
4,como,79
5,baños,32
6,disfrutan,32
7,invita,37
8,famosas,47
9,playas,47


Con estas frecuencias calculamos el valor de tf.

In [14]:
import math
tf = {}
for palabra, frec in indice_invertido.items():
    tf[palabra] = frec/len(contenido)

Tabla_tf = pd.DataFrame(list(tf.items()), columns=["palabra", "tf"])
Tabla_tf.head(20)

,palabra,tf
0,avistamiento,0.001064
1,laguna,0.000774
2,sorprende,0.000798
3,patrimonio,0.000484
4,como,0.001911
5,baños,0.000774
6,disfrutan,0.000774
7,invita,0.000895
8,famosas,0.001137
9,playas,0.001137


Con el valor de tf ya calculado, procedemos a calcular el valor de idf. Para este primer caso, hay que considerar el número total de términos y la frecuencia con que aparecen dentro del documento. Esto debido a que estamos evaluando un solo libro, caso contrario tendríamos un valor de cero para el idf indicando que el término no es muy informativo al compararlo en un solo libro.

In [15]:
N = len(contenido.split())
idf = {}
for palabra, frec in indice_invertido.items():
    df_t = frec
    idf[palabra] = math.log10(N/df_t)
    
Tabla_idf = pd.DataFrame(list(idf.items()), columns=["palabra", "idf"])
Tabla_idf.head(20)    

,palabra,idf
0,avistamiento,2.142660
1,laguna,2.280962
2,sorprende,2.267598
3,patrimonio,2.485082
4,como,1.888485
5,baños,2.280962
6,disfrutan,2.280962
7,invita,2.217911
8,famosas,2.114014
9,playas,2.114014


Creamos la matriz TF-IDF para este único libro.

In [16]:
TF_IDF = {}
for palabra in palabras:
    TF_IDF[palabra] = tf[palabra] * idf[palabra]

Tabla_TF_IDF = pd.DataFrame(list(TF_IDF.items()), columns=["palabra", "TF-IDF"])
Tabla_TF_IDF.head(20)

,palabra,TF-IDF
0,otavalo,0.001810
1,es,0.004923
2,conocido,0.001810
3,por,0.004985
4,su,0.007133
5,mercado,0.001810
6,indígena,0.001810
7,y,0.006094
8,artesanía,0.001810
9,perfecto,0.004320


Realizando con la librería sklearn

### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

In [1]:
import re


path2 = "..\\Data\\Gutenberg_1000"
files = os.listdir(path2)
contenidos_libros = []
nombres_libros = []
indice_invertido = {}
for file in files:
    # Se iteran los archivos
    with open(path2 + "\\" + file, "r", encoding="utf-8") as f:
        contenido = f.read().lower()
        # Extrae solo palabras (letras y números)
        palabras = re.findall(r'\w+', contenido)
        contenidos_libros.append(contenido)
        nombres_libros.append(file)
        vocabulario = set(palabras)
        for palabra in vocabulario:
            if palabra in indice_invertido:
                indice_invertido[palabra].append(file)
            else:
                indice_invertido[palabra] = [file]
print('Número de palabras del índice invertido: ',len(indice_invertido))
print(f"✓ Libros cargados: {len(contenidos_libros)}")


NameError: name 'os' is not defined

In [21]:
Tabla_archivos = pd.DataFrame(list(indice_invertido.items()), columns=['Palabra', 'Archivos'])
Tabla_archivos.head(20)

,Palabra,Archivos
0,deposition,"[Book1.txt, book10392.txt, book10531.txt, book..."
1,subsist,"[Book1.txt, book10.txt, book10077.txt, book103..."
2,equipment,"[Book1.txt, book10.txt, book100.txt, book10008..."
3,contemplate,"[Book1.txt, book10.txt, book10008.txt, book103..."
4,subsequent,"[Book1.txt, book10.txt, book10058.txt, book100..."
5,irrevocably,"[Book1.txt, book10916.txt, book11.txt, book117..."
6,narrowed,"[Book1.txt, book10.txt, book10429.txt, book105..."
7,stings,"[Book1.txt, book10447.txt, book10457.txt, book..."
8,pocket,"[Book1.txt, book10.txt, book100.txt, book10008..."
9,changes,"[Book1.txt, book10.txt, book100.txt, book10008..."


### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

Primero importamos las librerías necesarias.

In [20]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer

Con el contenido pre-cargado realizamos el cálculo de la mmatriz TF-IDF.

In [22]:
# Crear el vectorizador TF-IDF para Gutenberg 1000
# - sublinear_tf=True: TF logarítmico (1 + log(tf)) para amortiguar términos muy frecuentes
# - min_df=2:          ignorar términos que aparecen en menos de 2 libros
# - max_df=0.85:       ignorar términos en más del 85% de los libros (palabras muy comunes)
# - max_features=50000: limitar el vocabulario a los 50.000 términos más frecuentes
# - stop_words='english': eliminar stopwords en inglés
vectorizador_gutenberg = TfidfVectorizer(
    sublinear_tf=True,
    min_df=2,
    max_df=0.85,
    max_features=50000,
    stop_words="english",
    encoding="utf-8",
    decode_error="replace"
)

start_time = time.time()
print("Calculando matriz TF-IDF para Gutenberg 1000...")

# fit_transform aprende el vocabulario y construye la matriz (libros x términos)
matriz_tfidf_gutenberg = vectorizador_gutenberg.fit_transform(contenidos_libros)

elapsed = time.time() - start_time
print(f"✓ Matriz calculada en {elapsed:.2f} segundos")
print(f"\nDimensiones de la matriz TF-IDF (Gutenberg 1000):")
print(f"  → {matriz_tfidf_gutenberg.shape[0]} documentos  x  {matriz_tfidf_gutenberg.shape[1]} términos únicos")
print(f"\nTamaño en memoria (sparse): {matriz_tfidf_gutenberg.data.nbytes / 1024 / 1024:.2f} MB")
print(f"Elementos no cero: {matriz_tfidf_gutenberg.nnz:,}")

Calculando matriz TF-IDF para Gutenberg 1000...
✓ Matriz calculada en 65.16 segundos

Dimensiones de la matriz TF-IDF (Gutenberg 1000):
  → 1000 documentos  x  50000 términos únicos

Tamaño en memoria (sparse): 37.29 MB
Elementos no cero: 4,887,463


In [23]:
# Visualizar estadísticas de la matriz
terminos_gutenberg = vectorizador_gutenberg.get_feature_names_out()

# Top 20 términos con mayor IDF (los más raros / específicos)
import numpy as np
idf_values = vectorizador_gutenberg.idf_
top_idf_idx = np.argsort(idf_values)[::-1][:20]

print("Top 20 términos con mayor IDF (más específicos del corpus):")
for idx in top_idf_idx:
    print(f"  {terminos_gutenberg[idx]:<25} IDF = {idf_values[idx]:.4f}")

# Top 20 términos con menor IDF (los más comunes)
bot_idf_idx = np.argsort(idf_values)[:20]
print("\nTop 20 términos con menor IDF (más comunes):")
for idx in bot_idf_idx:
    print(f"  {terminos_gutenberg[idx]:<25} IDF = {idf_values[idx]:.4f}")

Top 20 términos con mayor IDF (más específicos del corpus):
  一卷                        IDF = 6.8101
  bolsheviki                IDF = 6.8101
  bondel                    IDF = 6.8101
  ripoll                    IDF = 6.8101
  ών                        IDF = 6.8101
  σωκράτη                   IDF = 6.8101
  υπέρ                      IDF = 6.8101
  morisque                  IDF = 6.8101
  blondine                  IDF = 6.8101
  bluegum                   IDF = 6.8101
  bracknell                 IDF = 6.8101
  bowne                     IDF = 6.8101
  potassii                  IDF = 6.8101
  bourienne                 IDF = 6.8101
  porthos                   IDF = 6.8101
  bruoder                   IDF = 6.8101
  brownlows                 IDF = 6.8101
  bruck                     IDF = 6.8101
  broadley                  IDF = 6.8101
  brocklehurst              IDF = 6.8101

Top 20 términos con menor IDF (más comunes):
  years                     IDF = 1.1623
  know                      IDF =

### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

In [30]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def buscar(consulta: str, top_n: int = 10) -> pd.DataFrame:
    """
    Busca los documentos más relevantes para una consulta usando el modelo vectorial TF-IDF.

    El proceso es:
      1. Transformar la consulta al espacio vectorial TF-IDF con el mismo vocabulario
         que fue aprendido al construir la matriz (fit_transform del corpus).
      2. Calcular la similitud coseno entre el vector de la consulta y todos los documentos.
      3. Retornar los top_n documentos ordenados por similitud descendente.

    Parámetros:
        consulta (str): Texto de búsqueda libre.
        top_n   (int): Número de resultados a retornar (por defecto 10).

    Retorna:
        pd.DataFrame con columnas [rank, libro, similitud]
    """
    start = time.time()

    # Transformar la consulta usando el vectorizador ya ajustado (solo transform, no fit)
    vector_consulta = vectorizador_gutenberg.transform([consulta])

    # Calcular similitud coseno entre la consulta y todos los documentos
    # cosine_similarity retorna un array de forma (1, n_documentos)
    similitudes = cosine_similarity(vector_consulta, matriz_tfidf_gutenberg).flatten()

    # Obtener los índices de los top_n documentos más similares
    top_indices = np.argsort(similitudes)[::-1][:top_n]

    elapsed = time.time() - start

    # Construir el DataFrame de resultados
    resultados = pd.DataFrame({
        "rank":       range(1, len(top_indices) + 1),
        "libro":      [nombres_libros[i] for i in top_indices],
        "similitud":  [round(similitudes[i], 6) for i in top_indices]
    })

    print(f"Consulta: '{consulta}'")
    print(f"Tiempo de búsqueda: {elapsed:.4f} segundos")
    print(f"Documentos con similitud > 0: {np.sum(similitudes > 0)}")
    return resultados

### Prueba 1: búsqueda de un tema específico
----

In [31]:
resultados = buscar("whale sea adventure ocean voyage", top_n=10)
print(resultados.to_string(index=False))

Consulta: 'whale sea adventure ocean voyage'
Tiempo de búsqueda: 0.4951 segundos
Documentos con similitud > 0: 789
 rank         libro  similitud
    1 book11105.txt   0.120206
    2 book57091.txt   0.059361
    3 book41221.txt   0.052793
    4 book44413.txt   0.052046
    5 book34392.txt   0.048929
    6 book52462.txt   0.047420
    7 book10339.txt   0.047359
    8 book13442.txt   0.043886
    9     Book2.txt   0.043418
   10 book55479.txt   0.041350


### Prueba 2: búsqueda de amor y romance
----

In [32]:
resultados2 = buscar("love marriage romance society", top_n=10)
print(resultados2.to_string(index=False))

Consulta: 'love marriage romance society'
Tiempo de búsqueda: 0.2098 segundos
Documentos con similitud > 0: 786
 rank         libro  similitud
    1 book47697.txt   0.048561
    2 book11970.txt   0.042120
    3    book17.txt   0.041637
    4 book51945.txt   0.040890
    5 book14679.txt   0.040551
    6 book12038.txt   0.040169
    7    book26.txt   0.039984
    8 book40006.txt   0.039486
    9 book12563.txt   0.038783
   10 book11235.txt   0.038637


### Prueba 3: búsqueda de ciencia y tecnología
----

In [33]:
resultados3 = buscar("science experiment laboratory discovery", top_n=10)
print(resultados3.to_string(index=False))

Consulta: 'science experiment laboratory discovery'
Tiempo de búsqueda: 0.2073 segundos
Documentos con similitud > 0: 713
 rank         libro  similitud
    1 book42626.txt   0.081721
    2 book51231.txt   0.075974
    3 book49377.txt   0.057133
    4    book11.txt   0.056561
    5 book50783.txt   0.055242
    6 book41375.txt   0.046373
    7 book59706.txt   0.044531
    8  book3990.txt   0.042636
    9 book54609.txt   0.042319
   10 book10008.txt   0.041702
